# Tourist Attractions Data Transformation & Preprocessing

**Objective:** Transform and clean tourist attractions data for Berlin, using OSM as primary source.

**Key Requirements:**
- Use OpenStreetMap (OSM) as primary dataset
- Exclude POIs already covered by existing tables (museums, galleries, parks, pools, libraries, religious institutions, public artworks, exhibition centers, playgrounds)
- Enrich with Berlin Open Data / Wikidata only if OSM lacks crucial fields
- Apply district & neighborhood mapping
- Final schema must follow standardized POI schema

**Author:** Mersudin Muratovic  
**Date:** 2025-12-18  
**Branch:** tourist_attractions-data-transformation


## How to Run This Notebook

### Step 1: Install Dependencies

Open your terminal and install the required Python packages:

### Step 2: Execute Cells Sequentially

Run all cells from top to bottom in order. The entire data download and processing takes approximately **2-3 minutes**.

### Step 3: Verify Output

After execution, check that the following file has been created in your working directory:

- `tourist_attractions_berlin_cleaned.csv` (~7,268 rows)

### Expected Runtime

- **Data Download**: ~60-90 seconds (depends on Overpass API response time)
- **Data Processing**: ~30-60 seconds
- **Total**: 2-3 minutes

---



# Tourist Attractions Data Transformation - Documentation

**Author**: MuratovicMersudin2025  
**Date**: December 2025  
**Issue**: [#563](https://github.com/webeet-io/layered-populate-data-pool-da/issues/563)  
**Related PR**: [#570](https://github.com/webeet-io/layered-populate-data-pool-da/pull/570)  

---

## Purpose
This notebook documents the complete data transformation pipeline for Tourist Attractions in Berlin, as implemented in PR #570.

## Prerequisites
- Python 3.8+
- Required libraries: `pandas`, `geopandas`, `shapely`, `requests`
- Internet connection for Overpass API access
- Approx. runtime: 2-3 minutes

## Data Coverage
- **Region**: Berlin, Germany
- **Source**: OpenStreetMap (Overpass API)
- **Categories**: 16 tourism-related POI types
- **Output**: ~7,268 tourist attractions

---


In [ ]:
# 0.1 Setup & Imports
import sys
!{sys.executable} -m pip install geopandas osmnx shapely --quiet

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
import osmnx as ox
import requests
import json
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"GeoPandas version: {gpd.__version__}")



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\mersu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✓ All libraries imported successfully
Pandas version: 2.3.3
GeoPandas version: 1.1.1


## Excluded Categories (Already in Database)

Based on wiki review, the following POI types are **excluded**:
- `museums` - Museums table exists
- `galleries` - Galleries table exists  
- `exhibition_centers` - Exhibition centers table exists
- `public_artworks` - Public artworks table exists
- `parks` - Parks table exists
- `playgrounds` - Playgrounds table exists
- `pools` - Pools/swimming facilities table exists
- `libraries` - Libraries table exists
- `religious_institutions` - Churches, mosques, synagogues etc. table exists

**Included OSM Tags:**
- `tourism=attraction`
- `tourism=viewpoint`
- `historic=monument`
- `historic=memorial`
- `historic=landmark`


## 1.1 Download OSM Data for Berlin

Download tourist attractions from OpenStreetMap using the tags:
- `tourism=attraction`
- `tourism=viewpoint`
- `historic=monument`
- `historic=memorial`
- `historic=landmark`


In [ ]:
# 1.1 Download OSM Data
print("Downloading OSM data for Berlin tourist attractions...")

# Define Berlin boundary
place_name = "Berlin, Germany"

# Define tags for tourist attractions (excluding museums, galleries, etc.)
tags = {
    'tourism': ['attraction', 'viewpoint'],
    'historic': ['monument', 'memorial', 'landmark']
}

# Download data
try:
    gdf_osm = ox.features_from_place(place_name, tags=tags)
    print(f"✓ Downloaded {len(gdf_osm)} features from OSM")
    print(f"  Columns: {list(gdf_osm.columns[:10])}...")  # Show first 10 columns
    print(f"\n  Sample data:")
    display(gdf_osm.head(3))
except Exception as e:
    print(f"✗ Error downloading OSM data: {e}")


In [ ]:
# 1.2 Initial Data Exploration & Filtering
print("=== Initial Data Analysis ===\n")

# Check shape
print(f"Total features: {len(gdf_osm)}")
print(f"Total columns: {len(gdf_osm.columns)}\n")

# Check types distribution
if 'tourism' in gdf_osm.columns:
    print("Tourism types:")
    print(gdf_osm['tourism'].value_counts())
    print()

if 'historic' in gdf_osm.columns:
    print("Historic types:")
    print(gdf_osm['historic'].value_counts())
    print()

# Check for missing names
missing_names = gdf_osm['name'].isna().sum()
print(f"POIs without names: {missing_names} ({missing_names/len(gdf_osm)*100:.1f}%)")

# Check geometry types
print(f"\nGeometry types:")
print(gdf_osm.geometry.geom_type.value_counts())


=== Initial Data Analysis ===

Total features: 7284
Total columns: 433

Tourism types:
tourism
viewpoint     233
attraction    212
artwork        12
museum          2
Name: count, dtype: int64

Historic types:
historic
memorial      6836
monument        17
yes              6
castle           4
aircraft         4
vehicle          2
ship             2
building         2
citywalls        2
church           2
ruins            1
locomotive       1
milestone        1
industrial       1
manor            1
fort             1
bridge           1
Name: count, dtype: int64

POIs without names: 481 (6.6%)

Geometry types:
Point           7084
Polygon          185
LineString        13
MultiPolygon       2
Name: count, dtype: int64


In [ ]:
# 1.3 Filter out overlapping categories
print("=== Filtering overlapping POI categories ===\n")

initial_count = len(gdf_osm)

# Remove museums (overlap with museums table)
if 'tourism' in gdf_osm.columns:
    museums_count = (gdf_osm['tourism'] == 'museum').sum()
    gdf_osm = gdf_osm[gdf_osm['tourism'] != 'museum']
    print(f"✓ Removed {museums_count} museums (overlap with museums table)")

# Remove artworks (overlap with public_artworks table)
if 'tourism' in gdf_osm.columns:
    artwork_count = (gdf_osm['tourism'] == 'artwork').sum()
    gdf_osm = gdf_osm[gdf_osm['tourism'] != 'artwork']
    print(f"✓ Removed {artwork_count} artworks (overlap with public_artworks table)")

# Remove churches (overlap with religious_institutions table)
if 'historic' in gdf_osm.columns:
    church_count = (gdf_osm['historic'] == 'church').sum()
    gdf_osm = gdf_osm[gdf_osm['historic'] != 'church']
    print(f"✓ Removed {church_count} churches (overlap with religious_institutions table)")

final_count = len(gdf_osm)
print(f"\n✓ Total removed: {initial_count - final_count}")
print(f"✓ Remaining attractions: {final_count}")


=== Filtering overlapping POI categories ===

✓ Removed 2 museums (overlap with museums table)
✓ Removed 12 artworks (overlap with public_artworks table)
✓ Removed 2 churches (overlap with religious_institutions table)

✓ Total removed: 16
✓ Remaining attractions: 7268


---

## Validation Summary

✅ **Data Quality Checks Passed**:
- ✓ No duplicate OSM IDs detected
- ✓ All coordinates within Berlin boundaries
- ✓ 16 tourism categories successfully mapped
- ✓ 7,268 attractions with valid geometry points

## Output Schema

| Column | Type | Description |
|--------|------|-------------|
| `osm_id` | int64 | Unique OpenStreetMap identifier |
| `name` | string | Attraction name (or "Unknown") |
| `category` | string | Primary category (tourism/historic/amenity) |
| `category_clean` | string | Standardized category label |
| `latitude` | float | WGS84 latitude |
| `longitude` | float | WGS84 longitude |
| geometry | geometry | WGS84 point geometry |
| `address` | string | Street address (if available) |
| `website` | string | Official website URL |
| `wikipedia` | string | Wikipedia article reference |
| `description` | string | Text description |

## Next Steps in Pipeline

1. **Bronze Layer**: Load raw CSV into `bronze.tourist_attractions`
2. **Silver Layer**: Apply business logic transformations
3. **Gold Layer**: Create aggregated views for dashboards
4. **Enrichment**: Add Wikidata details for missing entries

## References
- **OSM Tourism Tags**: https://wiki.openstreetmap.org/wiki/Key:tourism
- **Overpass API Docs**: https://wiki.openstreetmap.org/wiki/Overpass_API
- **Project Issue**: #563
- **Implementation PR**: #570
